# License Plate Detection — Model 1: YOLOv8
**CMPS 261 — Machine Learning Project**

YOLOv8 is the recommended baseline. It is a single-stage detector — it predicts bounding boxes and class probabilities in one forward pass, making it very fast.

## 1. Prepare the Data

In [ ]:
import sys
sys.path.append('..')
from src.prepare_data import prepare

yaml_path = prepare()
print('Dataset YAML:', yaml_path)

## 2. Train YOLOv8n (Nano — fastest, good baseline)

In [ ]:
from ultralytics import YOLO
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {"MPS" if torch.backends.mps.is_available() else "CPU"}')

# Load pretrained YOLOv8 nano
model = YOLO('yolov8n.pt')

results = model.train(
    data    = yaml_path,
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    project = '../models',
    name    = 'yolov8n',
    exist_ok= True,
    verbose = True,
)

## 3. Validate on Validation Set

In [ ]:
metrics = model.val()
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

## 4. Evaluate on Test Set

In [ ]:
import yaml
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

test_metrics = model.val(split='test')
print(f'Test mAP@0.5      : {test_metrics.box.map50:.4f}')
print(f'Test mAP@0.5:0.95 : {test_metrics.box.map:.4f}')
print(f'Test Precision    : {test_metrics.box.mp:.4f}')
print(f'Test Recall       : {test_metrics.box.mr:.4f}')

## 5. Visualise Predictions on Test Images

In [ ]:
import os, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

test_img_dir = '../data/yolo/images/test'
test_images  = random.sample(os.listdir(test_img_dir), 8)

best_weights = '../models/yolov8n/weights/best.pt'
model_best   = YOLO(best_weights)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, fname in zip(axes, test_images):
    img_path = os.path.join(test_img_dir, fname)
    result   = model_best.predict(img_path, conf=0.25, verbose=False)[0]
    img      = Image.open(img_path).convert('RGB')
    ax.imshow(img)

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f'{conf:.2f}', color='lime', fontsize=8,
                bbox=dict(facecolor='black', alpha=0.4, pad=1))

    ax.set_title(fname, fontsize=7)
    ax.axis('off')

plt.suptitle('YOLOv8n — Predictions on Test Set', fontsize=13)
plt.tight_layout()
plt.savefig('../results/yolo_predictions.png', dpi=150)
plt.show()
print('Saved: results/yolo_predictions.png')

## 6. Save Metrics for Comparison

In [ ]:
import json, os

yolo_metrics = {
    'model'        : 'YOLOv8n',
    'map50'        : round(test_metrics.box.map50, 4),
    'map50_95'     : round(test_metrics.box.map, 4),
    'precision'    : round(test_metrics.box.mp, 4),
    'recall'       : round(test_metrics.box.mr, 4),
}

os.makedirs('../results', exist_ok=True)
with open('../results/yolo_metrics.json', 'w') as f:
    json.dump(yolo_metrics, f, indent=2)

print('Metrics saved to results/yolo_metrics.json')
print(json.dumps(yolo_metrics, indent=2))